# Классификация: Heart Disease UCI (Cleveland)


In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
import random
import warnings
warnings.filterwarnings('ignore')

from sklearn.model_selection import train_test_split, StratifiedKFold, cross_validate
from sklearn.compose import ColumnTransformer
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import StandardScaler, OneHotEncoder
from sklearn.linear_model import LogisticRegression
from sklearn.ensemble import RandomForestClassifier
from sklearn.metrics import (classification_report, accuracy_score, roc_auc_score, 
                             confusion_matrix, ConfusionMatrixDisplay, 
                             precision_score, recall_score, f1_score,
                             precision_recall_curve, average_precision_score)

RANDOM_SEED = 47
np.random.seed(RANDOM_SEED)
random.seed(RANDOM_SEED)

plt.rcParams["figure.figsize"] = (12, 8)
sns.set_style("whitegrid")

## 1. Загрузка данных и первичный осмотр

In [ ]:
df = pd.read_csv("heart_cleveland_upload.csv")
print("Размер данных:", df.shape)
print("\nТипы по столбцам:\n", df.dtypes)
print("\nПропуски:\n", df.isnull().sum())
display(df.head())

## 2. Расширенный EDA

In [ ]:
# числовые признаки
chisl_stolb = ['age', 'trestbps', 'chol', 'thalach', 'oldpeak']
display(df[chisl_stolb].describe().T)

# распределение цели
plt.figure(figsize=(6, 4))
sns.countplot(data=df, x='condition', palette='Set2')
plt.title("Распределение целевой переменной (0 - здоров, 1 - болен)")
plt.show()
print(df['condition'].value_counts())

In [ ]:
# boxplot по классам
fig, axes = plt.subplots(2, 3, figsize=(18, 10))
axes = axes.flatten()
for i, col in enumerate(chisl_stolb):
    sns.boxplot(data=df, x='condition', y=col, ax=axes[i], palette='Set3')
    axes[i].set_title(f'{col} по классам')
axes[-1].axis('off')
plt.tight_layout()
plt.show()

In [ ]:
# countplot категорий
kat_stolb = ['sex', 'cp', 'fbs', 'restecg', 'exang', 'slope', 'ca', 'thal']
fig, axes = plt.subplots(2, 4, figsize=(20, 10))
axes = axes.flatten()
for i, col in enumerate(kat_stolb):
    sns.countplot(data=df, x=col, hue='condition', ax=axes[i], palette='Set1')
    axes[i].set_title(f'{col}')
plt.tight_layout()
plt.show()

In [ ]:
# корреляция числовых
stolb_korr = chisl_stolb + ['condition']
plt.figure(figsize=(10, 6))
sns.heatmap(df[stolb_korr].corr(), annot=True, cmap="coolwarm", fmt=".2f", center=0)
plt.title("Матрица корреляций (только числовые признаки)")
plt.show()

## 3. Feature Engineering и разделение данных


In [ ]:
# новые признаки
def dobavit_priznaki(dannye):
    df = dannye.copy()
    df['age_thalach'] = df['age'] * df['thalach']
    df['chol_age'] = df['chol'] / df['age']
    df['oldpeak_high'] = (df['oldpeak'] > 1.0).astype(int)
    df['thalach_high'] = (df['thalach'] > 150).astype(int)
    return df

df_s_priznakami = dobavit_priznaki(df)
print("Новые признаки добавлены:", [c for c in df_s_priznakami.columns if c not in df.columns])
display(df_s_priznakami.head(3))

X = df_s_priznakami.drop(columns=['condition'])
y = df_s_priznakami['condition']

# отделяем test
X_vrem, X_test, y_vrem, y_test = train_test_split(
    X, y, test_size=0.2, random_state=RANDOM_SEED, stratify=y
)
# отделяем val
x_tren, x_val, y_tren, y_val = train_test_split(
    X_vrem, y_vrem, test_size=0.25, random_state=RANDOM_SEED, stratify=y_vrem
)

print(f"\nTrain: {x_tren.shape}, Val: {x_val.shape}, Test: {X_test.shape}")
print(f"Доля больных в train: {y_tren.mean():.2f}, val: {y_val.mean():.2f}, test: {y_test.mean():.2f}")

## 4. Пайплайн предобработки

In [ ]:
# признаки по типам
kat_priznaki = ['cp', 'restecg', 'slope', 'thal', 'ca', 'sex', 'fbs', 'exang']
chisl_priznaki = ['age', 'trestbps', 'chol', 'thalach', 'oldpeak', 
                'age_thalach', 'chol_age', 'oldpeak_high', 'thalach_high']

# препроцессор
preprocessor = ColumnTransformer(
    transformers=[
        ('num', StandardScaler(), chisl_priznaki),
        ('cat', OneHotEncoder(drop='first', sparse_output=False, 
                              handle_unknown='ignore'), kat_priznaki)
    ])

print("Пайплайн предобработки создан.")
print(f"Числовые признаки ({len(chisl_priznaki)}): {chisl_priznaki}")
print(f"Категориальные признаки ({len(kat_priznaki)}): {kat_priznaki}")
print("\nhandle_unknown='ignore' — неизвестные категории в val/test будут кодироваться нулями, а не вызывать ошибку.")

## 5. Кросс-валидация (честное сравнение моделей)

In [ ]:
cv = StratifiedKFold(n_splits=5, shuffle=True, random_state=RANDOM_SEED)
metriki = ['accuracy', 'precision', 'recall', 'f1', 'roc_auc', 'average_precision']

# словарь моделей
modeli = {
    "LogisticRegression": Pipeline([
        ('prep', preprocessor),
        ('model', LogisticRegression(class_weight='balanced', max_iter=1000, random_state=RANDOM_SEED))
    ]),
    "RandomForest": Pipeline([
        ('prep', preprocessor),
        ('model', RandomForestClassifier(n_estimators=200, max_depth=5, 
                                         class_weight='balanced', random_state=RANDOM_SEED))
    ])
}

# кросс-валидация
rezult_cv = {}
for imya_modeli, pipe in modeli.items():
    ocenki = cross_validate(pipe, x_tren, y_tren, cv=cv, scoring=metriki, n_jobs=-1)
    rezult_cv[imya_modeli] = {m: ocenki[f'test_{m}'].mean() for m in metriki}

rezult_cv_df = pd.DataFrame(rezult_cv).T
display(rezult_cv_df.round(4))

# визуализация
rezult_cv_df.plot(kind='bar', figsize=(12, 6), colormap='Set2')
plt.title("Сравнение моделей по кросс-валидации (5-fold)")
plt.ylabel("Среднее значение метрики")
plt.xticks(rotation=0)
plt.legend(loc='lower right')
plt.tight_layout()
plt.show()

## 6. Подбор порога на валидационной выборке

In [ ]:
# функция подбора порога
def podobrat_porog(ist, veroyatnosti, min_tochnost=0.6):
    tochnosti, polnoty, porogi = precision_recall_curve(ist, veroyatnosti)
    luchshiy_porog, luchshaya_polnota = 0.5, 0.0
    for tochn, poln, porog in zip(tochnosti[:-1], polnoty[:-1], porogi):
        if tochn >= min_tochnost and poln > luchshaya_polnota:
            luchshaya_polnota, luchshiy_porog = poln, porog
    return luchshiy_porog, luchshaya_polnota

# обучаем на train, смотрим val
rezult_val = {}
for imya_modeli, pipe in modeli.items():
    pipe.fit(x_tren, y_tren)
    veroyat_val = pipe.predict_proba(x_val)[:, 1]
    
    # порог по умолчанию
    pred_05 = (veroyat_val >= 0.5).astype(int)
    # оптимальный порог
    porog, _ = podobrat_porog(y_val, veroyat_val, min_tochnost=0.6)
    pred_opt = (veroyat_val >= porog).astype(int)
    
    rezult_val[imya_modeli] = {
        'thresh': porog,
        'recall@0.5': recall_score(y_val, pred_05),
        'prec@0.5': precision_score(y_val, pred_05, zero_division=0),
        'recall@opt': recall_score(y_val, pred_opt),
        'prec@opt': precision_score(y_val, pred_opt, zero_division=0),
        'roc_auc': roc_auc_score(y_val, veroyat_val),
        'pr_auc': average_precision_score(y_val, veroyat_val)
    }

rezult_val_df = pd.DataFrame(rezult_val).T
display(rezult_val_df.round(3))
print("\nВыбранные пороги:")
for imya_modeli in rezult_val:
    print(f"  {imya_modeli}: {rezult_val[imya_modeli]['thresh']:.3f}")

## 7. Финальная оценка на тестовой выборке

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(14, 5))

for i, (imya_modeli, pipe) in enumerate(modeli.items()):
    veroyat_test = pipe.predict_proba(X_test)[:, 1]
    porog = rezult_val[imya_modeli]['thresh']
    pred_test = (veroyat_test >= porog).astype(int)
    
    print(f"\n{'='*50}")
    print(f"{imya_modeli} (порог = {porog:.3f})")
    print(f"{'='*50}")
    print(classification_report(y_test, pred_test, target_names=['Здоров', 'Болен']))
    print(f"ROC-AUC: {roc_auc_score(y_test, veroyat_test):.4f}")
    print(f"PR-AUC:  {average_precision_score(y_test, veroyat_test):.4f}")
    
    # матрица ошибок
    matrica_oshibok = confusion_matrix(y_test, pred_test)
    tn, fp, fn, tp = matrica_oshibok.ravel()
    print(f"\nTN={tn}, FP={fp}, FN={fn}, TP={tp}")
    print(f"Пропущено больных (FN): {fn} ← минимизируем!")
    
    ConfusionMatrixDisplay.from_predictions(
        y_test, pred_test, display_labels=['Здоров', 'Болен'],
        cmap='Blues' if i == 0 else 'Greens', ax=axes[i]
    )
    axes[i].set_title(f'{imya_modeli}\nпорог={porog:.3f}, Recall={recall_score(y_test, pred_test):.3f}')

plt.tight_layout()
plt.show()

## 8. Выводы
Во время выполнения лабораторной работы были выполнены следующие этапы: 
 - Проведён расширенный EDA: статистики, boxplot и countplot по признакам, корреляция только для числовых.
 - Добавлен Feature Engineering: age*thalach, chol/age, бинаризация oldpeak и thalach.
 - Все преобразования обёрнуты в Pipeline — утечки данных исключены.
 - Модели сравнены в равных условиях (оба с class_weight='balanced') через 5-fold CV — результаты стабильны и воспроизводимы.
 - Порог подобран на валидации, тест использован один раз.
 - Добавлены ROC-AUC и PR-AUC — информативнее accuracy при медицинских задачах.

Минимизация False Negatives достигается комбинацией class_weight='balanced' и снижением порога вероятности. Порог подбирается автоматически по критерию «максимизировать Recall при Precision ≥ 0.6».

При обнаружении сердечно-сосудистых заболеваний приоритет — Recall (чувствительность). Даже при Precision ~0.65–0.7 модель улавливает большинство больных, а ложноположительные случаи безопаснее, чем пропуск больного.